# Module 1 · Embeddings — In-class Lab A 🧪
## Subword tokenization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mago-cinv/course-template/blob/master/modules/01-embeddings/labs/in-class/lab-a-tokenization.ipynb)

This is **Mini-lab A**, split out from the combined in-class lab so it runs on its own. It pairs with **Lesson 1** (tokenization).

### How this lab works

**The machinery is given to you.** Every tokenizer, every training call, every helper function below is written, explained, and ready to run. You are *not* asked to look up which argument `BpeTrainer` takes — that is plumbing, and plumbing is not the lesson.

**What you write is the investigation.** Sections 🔬 **E1** and 🔬 **E2** hand you the same toolkit and ask you to *use* it: turn a knob and measure what changes, build a comparison the notebook didn't make, and defend a conclusion with numbers you produced. There is no single right answer — there is a defensible one.

Treat everything above the 🔬 sections as your **sandbox**: read it, run it, then take it apart.

Everything runs on CPU in seconds.


In [ ]:
# Setup — run me first (works on Colab AND locally)
# On Colab this installs the few extra libraries; locally the course venv
# (.venv, environment/requirements.txt) already has everything.
import sys

if "google.colab" in sys.modules:
    %pip install -q datasets tokenizers

print("Setup OK — running on", "Colab" if "google.colab" in sys.modules else "local Python")

In [ ]:
# Imports + seeds — fixed seeds make every run (and every student) identical
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(0)

from tokenizers import Tokenizer, models, trainers, pre_tokenizers

print("ready")

## 1 · The corpus (provided)

The first **2,000** texts of **AG News** (short news snippets: world / sports / business / sci-tech), lowercased. Small enough to train a tokenizer on in seconds; real enough to show word structure. The slice is deterministic (`select(range(2000))`) so everyone gets the same corpus.

In [ ]:
# Load the corpus: first 2,000 AG News training texts, lowercased
from datasets import load_dataset

ds = load_dataset("fancyzhx/ag_news", split="train").select(range(2000))
corpus = [t.lower() for t in ds["text"]] #default corpus is 2000 texts, lowercased

print(f"{len(corpus)} texts")
print("sample:", corpus[0][:120], "…")
assert len(corpus) == 2000

## 2 · Your toolkit (provided)

One factory and five helpers. **This cell is the sandbox** — everything you do in E1 and E2 is built from these.

| Tool | What it gives you |
|---|---|
| `train_tokenizer(kind, vocab_size)` | a trained tokenizer; `kind` ∈ `"bpe"`, `"unigram"`, `"wordpiece"`, `"word"` |
| `pieces(tok, word)` | how that tokenizer cuts one word, as a list |
| `avg_pieces_per_word(tok, words)` | mean pieces per word — the **fragmentation** number |
| `unk_rate(tok, words)` | fraction of words that collapse to `[UNK]` |
| `piece_logprobs(uni)` | a Unigram tokenizer's learned $\log p(s)$, as a dict |
| `log_prob_of(logp, segmentation)` | $\log P(\mathbf{s})$ for any list of pieces |

Every knob is named and defaulted here so you can change it without reading library docs.

In [ ]:
# ══ THE TOOLKIT ══ everything below is provided; read it, then use it in E1 / E2

SPECIALS = ["[UNK]", "[PAD]"]


def train_tokenizer(kind, vocab_size=5000, texts=None):
    """Train a tokenizer on `texts` (defaults to `corpus`).

    kind:
      "bpe"       — merge the most frequent adjacent pair, repeatedly (Lesson 1)
      "unigram"   — learn p(s) per piece; tokenizing = pick the most probable split
      "wordpiece" — like BPE but merges by likelihood gain; marks continuations with '##'
      "word"      — no subwords at all: one token per whole word, everything else [UNK]

    vocab_size is a CEILING for unigram/wordpiece, an exact target for bpe/word.
    """
    texts = corpus if texts is None else texts
    model, trainer = {
        "bpe": (
            models.BPE(unk_token="[UNK]"),
            trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIALS),
        ),
        "unigram": (
            models.Unigram(),
            trainers.UnigramTrainer(vocab_size=vocab_size, special_tokens=SPECIALS, unk_token="[UNK]"),
        ),
        "wordpiece": (
            models.WordPiece(unk_token="[UNK]"),
            trainers.WordPieceTrainer(vocab_size=vocab_size, special_tokens=SPECIALS),
        ),
        "word": (
            models.WordLevel(unk_token="[UNK]"),
            trainers.WordLevelTrainer(vocab_size=vocab_size, special_tokens=SPECIALS),
        ),
    }[kind]

    tok = Tokenizer(model)
    tok.pre_tokenizer = pre_tokenizers.Whitespace()  # split on whitespace/punctuation FIRST
    tok.train_from_iterator(texts, trainer)          # ...subword merges happen inside those chunks
    return tok


def pieces(tok, word):
    """How `tok` cuts `word`, as a list of strings."""
    return tok.encode(word).tokens


def avg_pieces_per_word(tok, words):
    """Mean number of pieces per word — the FRAGMENTATION number.
    1.0 = every word is a single token; higher = chopped finer."""
    return float(np.mean([len(pieces(tok, w)) for w in words]))


def unk_rate(tok, words):
    """Fraction of `words` that the tokenizer gives up on ([UNK] anywhere in the split)."""
    return float(np.mean([("[UNK]" in pieces(tok, w)) for w in words]))


def piece_logprobs(uni):
    """A Unigram tokenizer's learned parameters ARE probabilities: {piece: log p(s)}."""
    return {piece: lp for piece, lp in json.loads(uni.to_str())["model"]["vocab"]}


def log_prob_of(logp, segmentation):
    """log P(s) = Σ_k log p(s_k) — Lesson 1's eq. (3), in log space.
    Returns -inf if any piece is not in the vocabulary (an impossible segmentation)."""
    return sum(logp.get(p, -np.inf) for p in segmentation)


# Two ready-made probe sets you can use anywhere (or replace with your own).
CORPUS_WORDS = [w for w, _ in Counter(w for t in corpus for w in t.split()).most_common(300)]
MADE_UP_WORDS = ["tokenizationology", "beargator", "unfriendlily", "quantumly",
                 "hyperspeedster", "misunderstandably", "crocodilish", "reblogging"]

print(f"toolkit ready — {len(CORPUS_WORDS)} common corpus words, "
      f"{len(MADE_UP_WORDS)} made-up words available as probe sets")

## 3 · Worked example — subwords solve the out-of-vocabulary problem

A **word-level** tokenizer only knows words it saw in training. Anything else becomes `[UNK]` — the model is handed a shrug. A **subword** tokenizer can always fall back to smaller pieces, so nothing is ever completely unknown.

*Expected:* the made-up words split into several BPE pieces with no `[UNK]`; the word-level tokenizer punts on every one of them.

In [ ]:
# Two tokenizers, same corpus, same vocabulary budget — different granularity
bpe = train_tokenizer("bpe", vocab_size=5000)
word_tok = train_tokenizer("word", vocab_size=5000)

for w in ["players", "tokenizationology", "beargator"]:
    print(f"  {w!r:21}")
    print(f"      BPE        -> {pieces(bpe, w)}")
    print(f"      word-level -> {pieces(word_tok, w)}")

print(f"\non the made-up words: BPE [UNK]-rate = {unk_rate(bpe, MADE_UP_WORDS):.0%}, "
      f"word-level [UNK]-rate = {unk_rate(word_tok, MADE_UP_WORDS):.0%}")

## 4 · Worked example — the granularity ladder

Lesson 1's trade-off: the finer you chop, the **smaller** the vocabulary you need but the **longer** every sequence gets. Subwords sit deliberately in the middle.

In [ ]:
# One sentence, three granularities
sample = corpus[0]
print(f"as words     : {len(sample.split()):4d} tokens   (vocab would be 10^5–10^6, OOV everywhere)")
print(f"as BPE pieces: {len(pieces(bpe, sample)):4d} tokens   (vocab 5,000, OOV solved)")
print(f"as characters: {len(sample):4d} tokens   (vocab ~100, no OOV, longest sequences)")

## 5 · Worked example — Unigram: a different criterion for the same job

BPE builds its vocabulary by **replaying learned merges**. **Unigram** takes a different view: it learns a probability $p(s)$ for every piece and treats tokenizing as *inference* — of all the ways to cut a word into vocabulary pieces, pick the most probable one:

$$P(\mathbf{s}) = \prod_{k=1}^{K} p(s_k), \qquad \mathbf{s} = (s_1,\dots,s_K) \tag{3}$$

Same corpus, same vocabulary budget — only the **criterion** changes.

*Expected:* Unigram also avoids `[UNK]`, but at least one word is cut differently from BPE.

In [ ]:
# Same data, same budget, different criterion
uni = train_tokenizer("unigram", vocab_size=5000)

print(f"BPE vocab     = {bpe.get_vocab_size()}")
print(f"Unigram vocab = {uni.get_vocab_size()}  (vocab_size is a CEILING for Unigram, not a target)\n")

for w in ["players", "tokenizationology", "beargator"]:
    same = "same" if pieces(bpe, w) == pieces(uni, w) else "DIFFERENT"
    print(f"  {w!r:21} [{same}]")
    print(f"      BPE     -> {pieces(bpe, w)}")
    print(f"      Unigram -> {pieces(uni, w)}")

## 6 · Worked example — Unigram's parameters *are* probabilities

Unlike BPE's merge list, what Unigram learns is a number per piece: $\log p(s)$. That means you can **score** any segmentation, not just the one the tokenizer picked — sum the log-probs, and by eq. (3) that sum *is* $\log P(\mathbf{s})$.

Below: the chosen split vs. the all-single-characters fallback for the same word.

*Expected:* every $\log p(s) \le 0$ (probabilities are $\le 1$), and the chosen split scores **higher** (less negative) than the character-by-character one — fewer, more probable pieces win.

In [ ]:
# Unigram's learned parameters, and what they let us do
logp = piece_logprobs(uni)

print(f"{len(logp)} (piece, log p(s)) pairs — a few examples:")
for piece, lp in list(logp.items())[:10]:
    print(f"  {piece!r:12} log p(s) = {lp:8.3f}")

# Score two competing segmentations of the SAME word
w = "tokenizationology"
chosen = pieces(uni, w) #higher probability segmentation
fallback = list(w)  # one piece per character

print(f"\nscoring {w!r} two ways:")
print(f"  chosen     {chosen}")
print(f"     log P(s) = {log_prob_of(logp, chosen):.3f}")
print(f"  characters {fallback}")
print(f"     log P(s) = {log_prob_of(logp, fallback):.3f}")
print(f"\n→ the chosen split wins by "
      f"{log_prob_of(logp, chosen) - log_prob_of(logp, fallback):.2f} log-prob points")

---

## 🔬 Exercise E1 — How much does the vocabulary-size knob actually buy?

Everything above used `vocab_size=5000` because we told you to. **Is that a good number?** You now hold the knob. Find out.

A bigger vocabulary means fewer pieces per word (less fragmentation) but a bigger embedding table to train later. Somewhere there are diminishing returns. Your job is to *locate them with evidence*.

**Design and run the experiment:**

1. Train BPE at a range of vocabulary sizes — e.g. `[250, 500, 1000, 2000, 5000, 10000]`.
2. For each, measure `avg_pieces_per_word` on **two** different probe sets: `CORPUS_WORDS` (words the tokenizer saw) and `MADE_UP_WORDS` (words it never saw).
3. Plot both curves on one axis — vocabulary size on x, fragmentation on y.
4. Store your measurements in a dict called `sweep` so the check cell can see them.

**Then answer, in the markdown cell below:** where do the returns flatten? Do the two curves flatten at the *same* place — and if not, what does that tell you about choosing a vocabulary size for text you haven't seen yet?

> Nothing here needs a library you haven't been handed. `train_tokenizer` and `avg_pieces_per_word` are enough.

In [ ]:
# TODO: sweep vocab_size and measure fragmentation on BOTH probe sets
# TODO: build `sweep` = {vocab_size: (avg_on_corpus_words, avg_on_made_up_words), ...}
# TODO: then plot the two curves on one axis and label them
# HINT: for v in sizes:  tok = train_tokenizer("bpe", vocab_size=v)
# HINT: avg_pieces_per_word(tok, CORPUS_WORDS) and avg_pieces_per_word(tok, MADE_UP_WORDS)
# HINT: plt.plot(sizes, [...], marker="o", label="..."); plt.xscale("log"); plt.legend()


In [ ]:
# Light check on YOUR experiment (in-class = completion, not correctness)
assert len(sweep) >= 4, "sweep at least 4 vocabulary sizes so a trend is visible"
assert all(isinstance(v, tuple) and len(v) == 2 for v in sweep.values()), \
    "each entry should be (fragmentation_on_seen, fragmentation_on_unseen)"

lo, hi = min(sweep), max(sweep)
assert sweep[hi][0] <= sweep[lo][0], "a bigger vocabulary should not fragment seen words MORE"
print(f"✓ swept {len(sweep)} vocabulary sizes")
print(f"  seen words   : {sweep[lo][0]:.2f} -> {sweep[hi][0]:.2f} pieces/word  ({lo:,} -> {hi:,} vocab)")
print(f"  unseen words : {sweep[lo][1]:.2f} -> {sweep[hi][1]:.2f} pieces/word")
print(f"  the gap between them {'widened' if (sweep[hi][1]-sweep[hi][0]) > (sweep[lo][1]-sweep[lo][0]) else 'narrowed'} as vocabulary grew")

**Your reading of the curves.** Where do the returns flatten, and do both curves flatten together? What would you pick for `vocab_size`, and why?

> TODO: your answer here (3–4 sentences)


---

## 🔬 Exercise E2 — Build your own tokenizer face-off

Section 5 compared BPE and Unigram on **three words we chose**. Three words is an anecdote, not evidence — and we only used two of the four available algorithms.

**Build the comparison properly, on words you care about.**

1. **Write your own probe set** — at least **12 words**, all of one kind you find interesting. Pick a category and commit to it: medical or legal jargon, Spanish words, brand names, proper nouns, very long compounds, misspellings, code identifiers… Something the AG News corpus would *not* have covered well.
2. **Run all three subword tokenizers on it** — `"bpe"`, `"unigram"`, `"wordpiece"` — trained at the same `vocab_size` so the comparison is fair.
3. **Quantify, don't eyeball.** For each: `avg_pieces_per_word` and `unk_rate` on *your* words. Print a comparison table.
4. **Measure disagreement:** on what fraction of your words do BPE and Unigram produce *different* splits? Print a few of those disagreements side by side.
5. Store the per-tokenizer numbers in a dict called `faceoff`.

**Then answer below:** which tokenizer was gentlest on your words, and did anything surprise you?

> Again — no new libraries. `train_tokenizer`, `pieces`, `avg_pieces_per_word` and `unk_rate` cover all of it.

In [ ]:
# TODO: 1) write MY_WORDS — 12+ words of one interesting category (say which!)
# TODO: 2) train "bpe", "unigram", "wordpiece" at the same vocab_size
# TODO: 3) build `faceoff` = {kind: {"avg_pieces": ..., "unk_rate": ...}} and print a table
# TODO: 4) measure how often BPE and Unigram DISAGREE on your words; show a few
# HINT: pieces(tok, w) gives one word's split; compare the two lists directly


In [ ]:
# Light check on YOUR face-off (in-class = completion, not correctness)
assert len(MY_WORDS) >= 12, "use at least 12 words so the averages mean something"
assert len(set(MY_WORDS)) == len(MY_WORDS), "no duplicate words"
assert set(faceoff) >= {"bpe", "unigram", "wordpiece"}, "compare all three subword algorithms"
assert all("avg_pieces" in r and "unk_rate" in r for r in faceoff.values())

gentlest = min(faceoff, key=lambda k: faceoff[k]["avg_pieces"])
print(f"✓ {len(MY_WORDS)} words × {len(faceoff)} tokenizers compared")
print(f"  gentlest on your probe set: {gentlest} "
      f"({faceoff[gentlest]['avg_pieces']:.2f} pieces/word)")
print(f"  spread across algorithms: "
      f"{max(r['avg_pieces'] for r in faceoff.values()) - min(r['avg_pieces'] for r in faceoff.values()):.2f} pieces/word")

**Your findings.** Which algorithm handled your words most gently, how big was the spread between them, and what surprised you?

> TODO: your answer here (3–4 sentences)


---

## Wrap-up — what you actually did

You were handed a working tokenization sandbox and used it to produce two findings that were **not** in the notebook:

- **E1** — you located the point where a bigger vocabulary stops paying, and showed that the answer differs for text the tokenizer has seen versus text it hasn't. That gap is the reason vocabulary size is a *generalization* decision, not a compression one.
- **E2** — you built a probe set of your own and quantified three algorithms on it, replacing "these two words split differently" with a measured disagreement rate.

**Concept check (discuss aloud):**

1. Why can a subword tokenizer *never* emit `[UNK]` for a word written in the same alphabet it trained on?
2. BPE and Unigram both avoid `[UNK]` yet cut words differently. State each one's criterion in a sentence.
3. Your fragmentation numbers were much worse on unseen words. Name one concrete downstream cost of that.

**Next:** Mini-lab B — those pieces become **vectors**.